In [2]:
import pandas as pd
import numpy as np

def generate_customer_profiles(data_path):
    print("--- Starting Offline Customer Profiling ---")
    
    # 1. Load historical data
    df = pd.read_csv(data_path)
    
    # 2. Aggregate historical behavior per customer
    # We focus on how they reacted when orders were ACTUALLY delayed
    customer_stats = df.groupby('Customer_ID').agg(
        Total_Orders=('Order_ID', 'count'),
        Delayed_Orders=('Delay_Days', lambda x: (x > 0).sum()),
        Late_Complaints=('Is_Late_Complaint', 'sum'),
        Contract_Grace_Period=('Contract_Grace_Period_Days', 'first') # Assuming constant per customer
    ).reset_index()
    
    # 3. Calculate Tolerance Metrics
    # Tolerance Score = 1 - (Late Complaints / Delayed Orders)
    # If they never had a delayed order, we assume a neutral score of 0.5
    customer_stats['Complaint_Rate_When_Delayed'] = np.where(
        customer_stats['Delayed_Orders'] > 0,
        customer_stats['Late_Complaints'] / customer_stats['Delayed_Orders'],
        0.5 
    )
    
    customer_stats['Tolerance_Score'] = 1.0 - customer_stats['Complaint_Rate_When_Delayed']
    
    # 4. Define Baseline Max_Safe_Delay_Days based on Tolerance and Contract
    def assign_baseline_delay(row):
        grace = row['Contract_Grace_Period']
        score = row['Tolerance_Score']
        
        if score > 0.90:       # High Tolerance: Very rarely complains
            return min(3, grace) 
        elif score > 0.70:     # Medium Tolerance
            return min(1, grace)
        else:                  # Low Tolerance / High Risk
            return 0           # Do not hold, dispatch immediately
            
    customer_stats['Base_Max_Safe_Delay_Days'] = customer_stats.apply(assign_baseline_delay, axis=1)
    
    # 5. Tag Customers for Business Dashboard
    customer_stats['Risk_Profile'] = pd.cut(
        customer_stats['Tolerance_Score'], 
        bins=[-np.inf, 0.7, 0.9, np.inf], 
        labels=['High Risk', 'Medium Risk', 'Low Risk']
    )
    
    # Save the profiles
    output_path = 'customer_risk_profiles.csv'
    customer_stats.to_csv(output_path, index=False)
    
    print(f"✅ Customer profiling complete. Profiles saved to {output_path}")
    print("\nSample Profiles:")
    print(customer_stats[['Customer_ID', 'Risk_Profile', 'Tolerance_Score', 'Base_Max_Safe_Delay_Days']].head())
    
    return customer_stats


profiles_df = generate_customer_profiles('processed_b2b_delivery_data.csv')

--- Starting Offline Customer Profiling ---
✅ Customer profiling complete. Profiles saved to customer_risk_profiles.csv

Sample Profiles:
  Customer_ID Risk_Profile  Tolerance_Score  Base_Max_Safe_Delay_Days
0   B2B-10100     Low Risk         1.000000                         3
1   B2B-10240    High Risk         0.000000                         0
2   B2B-10435     Low Risk         1.000000                         3
3   B2B-10589     Low Risk         1.000000                         3
4   B2B-10603  Medium Risk         0.833333                         1
